## Libraries

* Import necessary libraries

In [42]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTENC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

* Load data

In [9]:
data = pd.read_csv("/content/datasets/diabetes_cleaned.csv")
print(data.info())
print(data.duplicated().sum())
print(data.gender.unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96128 entries, 0 to 96127
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   gender               96128 non-null  object 
 1   age                  96128 non-null  float64
 2   hypertension         96128 non-null  int64  
 3   heart_disease        96128 non-null  int64  
 4   smoking_history      96128 non-null  object 
 5   bmi                  96128 non-null  float64
 6   HbA1c_level          96128 non-null  float64
 7   blood_glucose_level  96128 non-null  int64  
 8   diabetes             96128 non-null  int64  
dtypes: float64(3), int64(4), object(2)
memory usage: 6.6+ MB
None
0
['Female' 'Male']


* Separate Feature and Target variables

In [10]:
X = data.drop("diabetes", axis=1)
y = data["diabetes"]

In [11]:
X.dtypes

,0
gender,object
age,float64
hypertension,int64
heart_disease,int64
smoking_history,object
bmi,float64
HbA1c_level,float64
blood_glucose_level,int64


*Two variables, 'Gender' and 'Smoking history' to be encoded*

* Split data into train and test

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [13]:
# check on split data

print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(67289, 8)
(28839, 8)
diabetes
0    0.911769
1    0.088231
Name: proportion, dtype: float64
diabetes
0    0.911751
1    0.088249
Name: proportion, dtype: float64


* Separate feature types

In [14]:
categorical_features = ["gender", "smoking_history"]
numerical_features = ["age", "hypertension", "heart_disease", "bmi", "HbA1c_level", "blood_glucose_level"]

## Logistic Regression Model
------
(Baseline Model)

* Preprocessing

In [40]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

* Logistic Regression pipeline

In [16]:
log_reg = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

* Training and prediction

In [17]:
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)

* Accuracy check

In [18]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.9575227989874823


*A higher accuracy in this case does not necessarily indicate that the model performs well, due to the substantial class imbalance between diabetic and non-diabetic observations. Since the majority class consists of non-diabetic observations, the model can achieve relatively high accuracy by predicting more observations as non-diabetic, while potentially failing to correctly identify diabetic cases.*

In [19]:
# Confusion matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

# Classification report
print(classification_report(y_test, y_pred))

[[26006   288]
 [  937  1608]]
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     26294
           1       0.85      0.63      0.72      2545

    accuracy                           0.96     28839
   macro avg       0.91      0.81      0.85     28839
weighted avg       0.95      0.96      0.95     28839



*The baseline Logistic Regression model distinguishes non-diabetic cases very well, but its ability to detect diabetic cases is substantially weaker, with 63% recall.*

* Handling class imbalance

(Balanced Logistic Regression model)

In [20]:
log_reg_balanced = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        class_weight={0: 1, 1: 3},
        max_iter=1000
    ))
])

In [21]:
log_reg_balanced.fit(X_train, y_train)

y_pred = log_reg_balanced.predict(X_test)

In [22]:
def report (y_test, y_pred):
    accuracy = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    class_report = classification_report(y_test, y_pred)
    print(f" Accuracy: {accuracy}\n Confusion matrix: \n{cm}\n Classification report: \n{class_report}")

report(y_test, y_pred)


 Accuracy: 0.9414334755019245
 Confusion matrix: 
[[25252  1042]
 [  647  1898]]
 Classification report: 
              precision    recall  f1-score   support

           0       0.98      0.96      0.97     26294
           1       0.65      0.75      0.69      2545

    accuracy                           0.94     28839
   macro avg       0.81      0.85      0.83     28839
weighted avg       0.95      0.94      0.94     28839



*class_weight="balanced", and manually tuned class weight substantially increased the model's ability to detect class 1, but it also produced many more false positives.*

## Random Forest Classifier
------
(Baseline Model)

* Preprocessing

In [23]:
rf_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough"
)

* Random Forest pipeline

In [24]:
random_forest = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

* Train and Predict

In [25]:
random_forest.fit(X_train, y_train)
y_pred = random_forest.predict(X_test)

In [26]:
report(y_test, y_pred)


 Accuracy: 0.9690696626096605
 Confusion matrix: 
[[26197    97]
 [  795  1750]]
 Classification report: 
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     26294
           1       0.95      0.69      0.80      2545

    accuracy                           0.97     28839
   macro avg       0.96      0.84      0.89     28839
weighted avg       0.97      0.97      0.97     28839



*The baseline Random Forest has lower diabetic recall than the class-weighted Logistic Regression, despite having higher accuracy and F1.*

(Balanced Random Forest Model)

In [27]:
random_forest_balanced = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        class_weight={0: 1, 1: 3},
        n_estimators=100,
        random_state=42
    ))
])

In [28]:
random_forest_balanced.fit(X_train, y_train)
y_pred = random_forest_balanced.predict(X_test)

In [29]:
report(y_test, y_pred)

 Accuracy: 0.9683761572870072
 Confusion matrix: 
[[26178   116]
 [  796  1749]]
 Classification report: 
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     26294
           1       0.94      0.69      0.79      2545

    accuracy                           0.97     28839
   macro avg       0.95      0.84      0.89     28839
weighted avg       0.97      0.97      0.97     28839



*Among the models tested so far, the baseline Random Forest achieved the highest class-1 F1-score (0.80), highest precision (0.95), and highest overall accuracy (96.91%), while maintaining a 69% recall for diabetic cases.*

* Tuned Random Forest Model

In [30]:
param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 10, 20, 30, 40],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None]
}

In [31]:
rf_tuned = RandomizedSearchCV(
    estimator=random_forest,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1
)

* Fit, inspect parameters and predicion on tuned model

In [ ]:
rf_tuned.fit(X_train, y_train)

In [ ]:
print(rf_tuned.best_params_)
print(rf_tuned.best_score_)

rf_tuned_best = rf_tuned.best_estimator_
y_pred = rf_tuned_best.predict(X_test)

In [ ]:
report(y_test, y_pred)

*Hyperparameter tuning slightly improved the Random Forest's overall accuracy and precision, but had little effect on minority-class recall. Therefore, tuning the model's structural parameters alone did not adequately address the class imbalance, indicating the need for additional imbalance-handling techniques such as SMOTENC or further decision-threshold adjustment.*

## SMOTENC
---
* Class Imbalance Optimisation


In [33]:
print(y_train.value_counts())
print()
print(y_train.value_counts(normalize=True))

diabetes
0    61352
1     5937
Name: count, dtype: int64

diabetes
0    0.911769
1    0.088231
Name: proportion, dtype: float64


In [34]:
print(X.dtypes)

gender                  object
age                    float64
hypertension             int64
heart_disease            int64
smoking_history         object
bmi                    float64
HbA1c_level            float64
blood_glucose_level      int64
dtype: object


In [35]:
cat_features = ["gender", "smoking_history"]
smote_nc = SMOTENC(
    categorical_features=cat_features,
    random_state=42
)

X_train_smote, y_train_smote = smote_nc.fit_resample(X_train, y_train)

*Oversampling done on training data only.*

In [36]:
print(y_train_smote.value_counts())
print(y_train_smote.value_counts(normalize=True))

diabetes
0    61352
1    61352
Name: count, dtype: int64
diabetes
0    0.5
1    0.5
Name: proportion, dtype: float64


* Use balanced training data on baseline Random Forest model

In [37]:
random_forest_smote = Pipeline([
    ("preprocessor", rf_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

random_forest_smote.fit(X_train_smote, y_train_smote)
y_pred = random_forest_smote.predict(X_test)

In [38]:
report(y_test, y_pred)

 Accuracy: 0.9530496896563682
 Confusion matrix: 
[[25585   709]
 [  645  1900]]
 Classification report: 
              precision    recall  f1-score   support

           0       0.98      0.97      0.97     26294
           1       0.73      0.75      0.74      2545

    accuracy                           0.95     28839
   macro avg       0.85      0.86      0.86     28839
weighted avg       0.95      0.95      0.95     28839



*Applying SMOTENC to the Random Forest training data increased the recall for the minority diabetic class from 0.69 to 0.75, reducing false negatives from 795 to 645. However, this improvement came at the cost of lower precision, which decreased from 0.95 to 0.73, while false positives increased from 97 to 709. The F1-score also decreased from 0.80 to 0.74. This indicates that SMOTENC improved the model's ability to identify diabetic observations but did not improve the overall balance between precision and recall for this Random Forest configuration.*

## XGBClassifier
---
(Baseline Model)

In [43]:
xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    ))
])

xgb_pipeline.fit(X_train, y_train)

y_pred_xgb = xgb_pipeline.predict(X_test)

report(y_test, y_pred_xgb)

 Accuracy: 0.9703526474565692
 Confusion matrix: 
[[26210    84]
 [  771  1774]]
 Classification report: 
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     26294
           1       0.95      0.70      0.81      2545

    accuracy                           0.97     28839
   macro avg       0.96      0.85      0.89     28839
weighted avg       0.97      0.97      0.97     28839



*The baseline XGBoost model achieved high overall accuracy (97.04%) and strong minority-class precision (95%), while achieving a recall of 70% and F1-score of 81%. However, it still produced 771 false negatives, meaning that approximately 30% of the actual diabetic cases were not identified.*

(Balanced XGBClassifier)

In [44]:
xgb_balanced = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        random_state=42,
        scale_pos_weight=10.33,
        eval_metric="logloss"
    ))
])

xgb_balanced.fit(X_train, y_train)

y_pred_xgb_balanced = xgb_balanced.predict(X_test)

report(y_test, y_pred_xgb_balanced)

 Accuracy: 0.9189292277818232
 Confusion matrix: 
[[24276  2018]
 [  320  2225]]
 Classification report: 
              precision    recall  f1-score   support

           0       0.99      0.92      0.95     26294
           1       0.52      0.87      0.66      2545

    accuracy                           0.92     28839
   macro avg       0.76      0.90      0.80     28839
weighted avg       0.95      0.92      0.93     28839



*The class-balanced XGBoost substantially improved recall for the diabetic class from 0.70 to 0.87, reducing false negatives from 771 to 320. However, this came at a substantial cost to precision, which fell from 0.95 to 0.52, and F1-score, which fell from 0.81 to 0.66. The number of false positives also increased considerably, from 84 to 2,018.*

* Class Balance Optimisation (SMOTENC) on XGBClassifier

In [45]:
xgb_Smotenc = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        random_state=42,
        eval_metric="logloss"
    ))
])

xgb_Smotenc.fit(X_train_smote, y_train_smote)

y_pred_xgb_smotenc = xgb_Smotenc.predict(X_test)

report(y_test, y_pred_xgb_smotenc)

 Accuracy: 0.9671971982384965
 Confusion matrix: 
[[26100   194]
 [  752  1793]]
 Classification report: 
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     26294
           1       0.90      0.70      0.79      2545

    accuracy                           0.97     28839
   macro avg       0.94      0.85      0.89     28839
weighted avg       0.97      0.97      0.97     28839



*SMOTENC did not provide a meaningful improvement in minority-class recall for XGBoost in this experiment. Although it slightly reduced the number of false negatives, this came with an increase in false positives and a lower F1-score. This suggests that simply balancing the training data with synthetic observations was not sufficient to substantially improve XGBoost's detection of the minority class on this dataset.*

## Performance Comparison Summary
---
Model selection requires evaluating accuracy, precision, recall, F1-score, and the consequences of false positives and false negatives. The relative importance of these metrics depends on the intended use of the model.

To identify the best model, we must balance **Accuracy**, **Precision (Class 1)**, and **Recall (Class 1)**. In medical diagnostic scenarios like diabetes prediction, a false negative (failing to identify a diabetic patient) is typically much more costly than a false positive.

### Key Metrics Table
| Model | Accuracy | Class 1 Precision | Class 1 Recall | Class 1 F1-Score | False Negatives (FN) | False Positives (FP) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **Baseline Logistic Regression** | 95.75% | 0.85 | 0.63 | 0.72 | 937 | 288 |
| **Balanced Logistic Regression** | 94.14% | 0.65 | 0.75 | 0.69 | 647 | 1,042 |
| **Baseline Random Forest** | 96.91% | **0.95** | 0.69 | 0.80 | 795 | **97** |
| **Balanced Random Forest** | 96.84% | 0.94 | 0.69 | 0.79 | 796 | 116 |
| **SMOTENC + Random Forest** | 95.30% | 0.73 | 0.75 | 0.74 | 645 | 709 |
| **Baseline XGBoost** | **97.04%** | **0.95** | 0.70 | **0.81** | 771 | 84 |
| **Balanced XGBoost (scale_pos_weight)** | 91.89% | 0.52 | **0.87** | 0.66 | **320** | 2,018 |
| **SMOTENC + XGBoost** | 96.72% | 0.90 | 0.70 | 0.79 | 752 | 194 |

---

### Semantic Evaluation & Verdict

1. **Best Model for General Balance (Highest F1-Score): Baseline XGBoost**
   - **Why:** It yields the highest overall accuracy (97.04%) and the best F1-score (0.81) for the minority class, along with an outstanding precision of 0.95. If you want a reliable model that rarely flags healthy individuals incorrectly while still catching 70% of diabetic cases, this is the best baseline model.
   - *Best-performing model under a balanced evaluation emphasizing class-1 F1 while maintaining high precision.*

2. **Best Model for Safety-Critical Screening (Highest Recall): Balanced XGBoost**
   - **Why:** By setting `scale_pos_weight=10.33`, this model achieves a Recall of **87%**, reducing the false negatives dramatically from 771 down to just 320. However, this comes at the cost of precision dropping to 52% and generating 2,018 false positives.
   - *Best-performing model in this experiment when prioritizing minority-class recall.*

### Conclusion
- If your objective is a **screening tool** where catching every possible case of diabetes is critical, the **Balanced XGBoost** is the best model.
- If you need to avoid alarm fatigue and maintain high overall confidence in positive flags, the **Baseline XGBoost** is the superior, most stable model.